In [3]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv(override=True)

client = Anthropic()
model = "claude-sonnet-4-5"

In [1]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {
            "temperature": temperature,
            "thinking": {
                "type": "disabled"
            }
        },
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [7]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains":["nih.gov"]
}

In [8]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle based on recent studies? 
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_011CedRVaKq1g3aojMKiupeq', container=None, content=[ServerToolUseBlock(id='srvtoolu_01Xie7mweyfZUz7r9yXnWi8M', caller=None, input={'query': 'best exercise leg muscle growth recent studies 2026'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EuAQCioIExgCIiRmZTNkYjAxOS1hMjEyLTRkM2UtYmQ4Ny1mOGUyZGUzNjkyNDQSDCTmr66ivo9gbVPVQxoMgrrK8wdUCUxSYEW+IjAeXWY3G6o2AUUalxRuBSEQpwHm2B8sovALdpuGRZiK7juElZgZq9PhCsDLAJcKGBUq4w9enIDo5xp7D76NaR9Pe+PyvH7kWwMolH0szwvoLNnv4zBLmhZdSoSJ04GqPuC1V1AbR7oFRQfHpx42J1k7sYgYWAfE9DqGM/U07WPEF/TB9Z2QesT1EBDAH/A59rY5ZXde6l1K9skTldxYB5rUrnT9kzpD+mj2gnY+6Iat7p/uhzedYGB3o3gLG2M5vCQVwkOLSaJ1stQmtwrEADj6+0nIlCaPQHHep7sdvVOTSDbSnpDhqfl/4m/REn9CY1Mujfk0fzoBNxwIox3RX03ewCgqouyn7NBYKWIjcnn8HuQwzra0gya4qB7ciGszXkY2ijEaYxSi0hGQopW+6etzrc2NBoRiqQALw/7S2QahK53TVMD17Mi52k6TbnGCQOZcwDNRmFwiKckpt92EMd7muDwcPfZV6XpX+8sIIgpO42H+ZZLuSxf3rr5FuOeorGlUzoy3SZqpwGW3Fz+vM5